# 手续费 + 股息 + 隔夜利息 — 从零开始学 / Commission + Dividend + Swap — From Zero

交易不是免费的。每笔交易你可能要付三种费用：

Trading is not free. Each trade can carry up to three separate charges:

| 费用 / Fee | 什么时候收？ / When Charged | 谁收？ / Who Collects |
|------|-----------|-------|
| **手续费 (Commission)** | 开仓/平仓时 / On entry and exit | 平台 / The broker |
| **股息 (Dividend)** | 股票派息日 / On the stock's ex-dividend date | 市场（通过平台）/ The market (routed via the broker) |
| **隔夜利息 (Swap)** | 每晚持仓过夜 / Every night a position is held open | 平台 / The broker |

> 本 notebook 会逐一讲解这三种费用的计算方法，配合代码和图表。
>
> This notebook walks through the formula for each of the three charges with code and charts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
print('Setup done!')

---
## 第1节：手续费 (Commission) / Section 1: Commission

### 适用范围 / Applicability
手续费主要针对 **股票CFD**，外汇和指数的费用通常体现在点差中。

Commission mainly applies to **stock CFDs**. For FX and indices the cost is usually baked into the spread.

### 公式 / Formula

$$\boxed{\text{手续费 / Commission} = \text{手数 / Lots} \times \text{合约大小 / Contract} \times \text{开仓价格 / EntryPrice} \times \text{费率 / Rate}}$$

| 市场 / Market | 费率 / Rate | 合约大小 / Contract Size |
|------|------|--------|
| 美股 (AAPL, GOOGL, META, TSLA...) / US stocks | **0.10%** | 100股/手 / 100 sh per lot |
| 港股 (1810.HK 小米, HK50...) / HK stocks | **0.35%** | 100股/手 / 100 sh per lot |

### 记忆技巧 / Memory Tricks
- 美股便宜：**千分之一** (0.10% = 0.001) / US stocks are cheap — **one-tenth of a percent** (0.10% = 0.001)
- 港股贵3.5倍：**千分之三点五** (0.35% = 0.0035) / HK stocks are 3.5× pricier — **0.35%** (0.0035)

In [ ]:
# Commission calculator
def calc_commission(market, lots, contract, price):
    # US stocks 0.10%, HK stocks 0.35%
    rate = 0.001 if market == 'US' else 0.0035
    commission = lots * contract * price * rate
    ccy = 'USD' if market == 'US' else 'HKD'
    print(f'[{market} stock commission]')
    print(f'  = {lots} x {contract} x {price} x {rate}')
    print(f'  = {commission:.2f} {ccy}')
    return commission

print('=== Q1: META ===')
calc_commission('US', lots=1, contract=100, price=320.00)

print('\n=== Q3: GOOG ===')
calc_commission('US', lots=2, contract=100, price=172.00)

print('\n=== Q7: AMZN ===')
calc_commission('US', lots=5, contract=100, price=129.50)

print('\n=== Q11: Xiaomi (HK) ===')
hkd = calc_commission('HK', lots=10, contract=100, price=14.60)
print(f'  Convert to USD: {hkd:.2f} / 7.82 = {hkd / 7.82:.2f} USD')

In [ ]:
# Commission comparison chart
stocks = ['META\n$320', 'GOOG\n$172', 'AMZN\n$129.5', 'TSLA\n$785', 'NFLX\n$400', 'Xiaomi\nHKD14.6']
commissions = [
    1 * 100 * 320 * 0.001,            # META 1 lot
    2 * 100 * 172 * 0.001,            # GOOG 2 lots
    5 * 100 * 129.5 * 0.001,          # AMZN 5 lots
    3 * 100 * 785 * 0.001,            # TSLA 3 lots
    1 * 100 * 400 * 0.001,            # NFLX 1 lot
    10 * 100 * 14.6 * 0.0035 / 7.82,  # Xiaomi 10 lots (HKD->USD)
]
lots_info = ['1 lot', '2 lots', '5 lots', '3 lots', '1 lot', '10 lots']
colors = ['#3498db'] * 5 + ['#e74c3c']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(stocks, commissions, color=colors, edgecolor='black', linewidth=0.5)
for bar, val, lot in zip(bars, commissions, lots_info):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'${val:.1f}\n({lot})', ha='center', fontsize=9)
ax.set_ylabel('Commission (USD)', fontsize=12)
ax.set_title('Commission per question (blue = US 0.10%, red = HK 0.35% to USD)',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 第2节：股息 (Dividend) / Section 2: Dividend

当你持有股票CFD时，如果遇到派息日：

When you are holding a stock CFD on its ex-dividend date:

### 多单（买入）→ 你收到股息（扣1%税）/ Long → You Receive the Dividend (minus 1% tax)

$$\text{多单股息 / Long dividend} = (\text{每股股息 / DivPerShare} \times \text{合约量 / Contract} \times \text{手数 / Lots}) \times 99\%$$

### 空单（卖出）→ 你要付股息（加1%税）/ Short → You Pay the Dividend (plus 1% tax)

$$\text{空单股息 / Short dividend} = -(\text{每股股息 / DivPerShare} \times \text{合约量 / Contract} \times \text{手数 / Lots}) \times 101\%$$

### 为什么？ / Why?

| 方向 / Direction | 逻辑 / Logic | 1%税的含义 / What the 1% Tax Means |
|------|------|----------|
| 多单 / Long | 你"持有"股票 → 有权拿股息 / You "own" the stock → entitled to the dividend | 平台扣1%作为处理费 / Broker keeps 1% as a handling fee |
| 空单 / Short | 你"借"了别人的股票来卖 → 借的股票本来有股息 → 你要赔给人家 / You borrowed stock to short → the lender would have received the dividend → you must reimburse them | 你还要额外交1%的费 / You also pay an extra 1% fee |

> **简记**：多单赚99%，空单亏101%
>
> **Quick recall**: long earns 99%, short loses 101%.

In [ ]:
# Dividend calculator
def calc_dividend(direction, dividend_per_share, contract, lots, currency='USD', fx_rate=1):
    gross = dividend_per_share * contract * lots
    tax = 0.01 * gross
    if direction == 'long':
        result = gross - tax  # receive dividend - 1% tax
        sign = '+'
    else:
        result = -(gross + tax)  # pay dividend + 1% tax
        sign = ''

    print(f'  Gross dividend = {dividend_per_share} x {contract} x {lots} = {gross:.2f} {currency}')
    print(f'  1% tax fee     = {tax:.2f} {currency}')
    if direction == 'long':
        print(f'  {direction} receives: {gross:.2f} - {tax:.2f} = {sign}{result:.2f} {currency}')
    else:
        print(f'  {direction} pays:     -{gross:.2f} - {tax:.2f} = {result:.2f} {currency}')

    if fx_rate != 1:
        print(f'  Convert to USD: {result:.2f} / {fx_rate} = {result / fx_rate:.2f} USD')
    return result

print('=== Q1: META short, dividend $1.50 ===')
calc_dividend('short', 1.50, 100, 1)

print('\n=== Q3: GOOG long, dividend $2.10 ===')
calc_dividend('long', 2.10, 100, 2)

print('\n=== Q7: AMZN short, dividend $3.00 ===')
calc_dividend('short', 3.00, 100, 5)

print('\n=== Q10: TSLA short, dividend $1.75 ===')
calc_dividend('short', 1.75, 100, 3)

print('\n=== Q11: Xiaomi short, dividend HKD 0.40 ===')
calc_dividend('short', 0.40, 100, 10, currency='HKD', fx_rate=7.82)

print('\n=== Q14: NFLX short, dividend $4.00 ===')
calc_dividend('short', 4.00, 100, 1)

---
## 第3节：隔夜利息 (Swap / Overnight Interest) / Section 3: Overnight Swap

### 什么是隔夜利息？ / What Is Swap?

你用杠杆交易 = 平台借钱给你。持仓过夜 = 借钱过夜 = 要付利息。

Leveraged trading means the broker is lending you money. Holding overnight = borrowing overnight = paying interest.

### 三种计算模式 / Three Calculation Modes

| 模式 / Mode | 公式 / Formula | 适用于 / Applies To |
|------|------|-------|
| **点差模式 / Pip mode** | 手数 × 库存费 × 点值 × 天数 / Lots × SwapRate × PipValue × Days | 外汇（EURUSD, GBPUSD等）/ FX (EURUSD, GBPUSD, ...) |
| **百分比模式 / Percent mode** | (手数 × 价格 × 合约 × 费率% / 360) × 天数 / (Lots × Price × Contract × Rate% / 360) × Days | 部分品种 / Selected instruments |
| **期货模式 / Futures mode** | 手数 × 价格 × 合约 × (-1%) / 360 × 天数 / Lots × Price × Contract × (-1%) / 360 × Days | 加密货币(BTC,ETH)、指数 / Crypto (BTC, ETH) and indices |

### 关键概念：周三三倍利息 / Key Concept: Triple Swap on Wednesday

外汇交易的结算是 T+2（交易后2天结算）。周三晚上过夜 = 跨越了周末的结算日，所以收3天的利息。

FX settles T+2 (two days after trade date). Holding through Wednesday night means the settlement crosses the weekend, so you are charged three days of interest.

| 持仓过夜日 / Night Held | 收几天利息？ / Days Charged | 原因 / Reason |
|-----------|------------|------|
| 周一晚 / Monday night | 1天 / 1 day | 正常 / Normal |
| 周二晚 / Tuesday night | 1天 / 1 day | 正常 / Normal |
| **周三晚 / Wednesday night** | **3天 / 3 days** | 包含周六+周日 / Includes Sat + Sun |
| 周四晚 / Thursday night | 1天 / 1 day | 正常 / Normal |
| 周五晚 / Friday night | 1天 / 1 day | 正常 / Normal |
| **一周五晚合计 / Total for 5 weeknights** | **7天 / 7 days** | 1+1+3+1+1 |

In [ ]:
# Wednesday triple swap visualization
days = ['Mon\nnight', 'Tue\nnight', 'Wed\nnight', 'Thu\nnight', 'Fri\nnight']
swap_days = [1, 1, 3, 1, 1]
colors = ['#3498db', '#3498db', '#e74c3c', '#3498db', '#3498db']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(days, swap_days, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, swap_days):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val}d', ha='center', fontsize=14, fontweight='bold',
            color='red' if val == 3 else 'black')
ax.set_ylabel('Days charged', fontsize=12)
ax.set_title('Swap days per night of the week (Wed = 3 days!)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 4)
ax.axhline(y=1, color='gray', ls='--', alpha=0.3)
ax.text(2, 3.5, 'Wed = triple swap!\n(Sat + Sun rolled in)', ha='center', fontsize=11, color='red',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='red'))
plt.tight_layout()
plt.show()
print(f'Total for the week: {sum(swap_days)} days of interest (not 5!)')

### 3.1 点差模式 — 外汇 / 3.1 Pip Mode — FX

$$\text{隔夜利息 / Swap} = \text{手数 / Lots} \times \text{库存费(点) / SwapRate (pips)} \times \text{点值 / PipValue} \times \text{天数 / Days}$$

库存费（swap rate）由平台给出，通常是个负数（你付钱），单位是"点"。

The swap rate is set by the broker, usually negative (you pay), quoted in pips.

| 货币对 / Pair | 点值 / Pip Value | 库存费举例 / Sample Swap Rate | 每手每晚费用 / Cost per Lot per Night |
|--------|------|-----------|------------|
| EURUSD | $1/点 / $1 per pip | -4.23 | 4.23 × $1 = **$4.23** |
| GBPUSD | $1/点 / $1 per pip | -6.21 | 6.21 × $1 = **$6.21** |

In [ ]:
# Pip-mode swap calculation
def swap_pip_mode(pair, lots, swap_rate, pip_value, days, day_label=''):
    total = lots * swap_rate * pip_value * days
    print(f'[{pair} swap — pip mode]')
    print(f'  Formula: Lots x SwapRate x PipValue x Days')
    print(f'  = {lots} x ({swap_rate}) x {pip_value} x {days}{" (" + day_label + ")" if day_label else ""}')
    print(f'  = {total:.2f} USD')
    return total

print('=== Q2: EURUSD 3 lots long, 3 nights ===')
swap_pip_mode('EURUSD', lots=3, swap_rate=-4.23, pip_value=1, days=3, day_label='3 nights = 3 days')

print('\n=== Q9: GBPUSD 1 lot, 5 weeknights ===')
swap_pip_mode('GBPUSD', lots=1, swap_rate=-6.21, pip_value=1, days=7,
              day_label='5 nights = 7 days, includes Wed triple')

### 3.2 期货模式 — 加密货币 & 指数 / 3.2 Futures Mode — Crypto & Indices

对于 BTC、ETH 等加密货币和部分指数，使用固定 **年化 -1%** 的利率：

For BTC, ETH, and selected indices, a flat **-1% annualized** rate is used:

$$\text{隔夜利息 / Swap} = \frac{\text{手数 / Lots} \times \text{现价 / Price} \times \text{合约 / Contract} \times (-1\%)}{360} \times \text{天数 / Days}$$

拆解 / Broken down:
- `手数 × 现价 × 合约` / `Lots × Price × Contract` = 你的仓位名义价值 / your position's notional value
- `× (-1%)` = 年化利率 1% / an annual rate of 1%
- `÷ 360` = 换算成每天 / convert to a daily rate
- `× 天数` / `× Days` = 持仓几天 / times the number of nights held

In [ ]:
# Futures-mode swap calculation
def swap_futures_mode(product, lots, price, contract, days, day_label=''):
    notional = lots * price * contract
    daily = notional * 0.01 / 360
    total = -daily * days
    print(f'[{product} swap — futures mode]')
    print(f'  Notional = {lots} x {price:,.2f} x {contract} = {notional:,.2f}')
    print(f'  Daily interest = {notional:,.2f} x 1% / 360 = {daily:.4f}')
    print(f'  Total = -{daily:.4f} x {days}{" (" + day_label + ")" if day_label else ""} = {total:.2f} USD')
    return total

print('=== Q6: BTCUST 3 lots, Wed -> Fri ===')
swap_futures_mode('BTCUST', lots=3, price=98752, contract=1, days=4,
                  day_label='Wed night 3d + Thu night 1d = 4 days')

print('\n=== Q12: ETHUST 3 lots, Mon -> Fri ===')
swap_futures_mode('ETHUST', lots=3, price=2300, contract=1, days=6,
                  day_label='Mon1 + Tue1 + Wed3 + Thu1 = 6 days')

---
## 第4节：持仓天数怎么算？（常见题型）/ Section 4: Counting Days Held (Common Question Types)

记住一个核心规则：**周三晚 = 3天，其余每晚 = 1天**

Remember one core rule: **Wed night counts as 3 days, every other night counts as 1 day.**

| 题目说法 / Problem Description | 过了几晚？ / Nights Held | 库存费天数 / Swap Days | 分析 / Analysis |
|---------|----------|----------|------|
| 周二到周三 / Tue → Wed | 1晚（周二晚）/ 1 night (Tue) | **1天 / 1 day** | 普通晚 / Normal |
| 周三到周四 / Wed → Thu | 1晚（周三晚）/ 1 night (Wed) | **3天 / 3 days** | 周三三倍！/ Wed triple! |
| 周三到周五 / Wed → Fri | 2晚 / 2 nights | **4天 / 4 days** | 周三(3)+周四(1) / Wed(3) + Thu(1) |
| 周一到周五 / Mon → Fri | 4晚 / 4 nights | **6天 / 6 days** | 周一(1)+周二(1)+周三(3)+周四(1) / Mon(1) + Tue(1) + Wed(3) + Thu(1) |
| 周五到周一 / Fri → Mon | 1晚（周五晚）/ 1 night (Fri) | **3天 / 3 days** | 跨周末 / Spans the weekend |
| 一周五晚 / Full week, 5 weeknights | 5晚 / 5 nights | **7天 / 7 days** | 1+1+3+1+1 |

In [ ]:
# Day counter
swap_per_night = {'Mon': 1, 'Tue': 1, 'Wed': 3, 'Thu': 1, 'Fri': 1}
nights_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']

def count_swap_days(start, end):
    """Count swap days held from `start` weekday night to `end` weekday night."""
    start_idx = nights_order.index(start)
    end_idx = nights_order.index(end)

    if end_idx <= start_idx:  # wraps across the weekend
        end_idx += 5

    total = 0
    details = []
    for i in range(start_idx, end_idx):
        night = nights_order[i % 5]
        d = swap_per_night[night]
        details.append(f'{night} night ({d}d)')
        total += d

    print(f'  {start} -> {end}: {" + ".join(details)} = {total} days')
    return total

print('Days held for each question:')
print('Q6: ',  end=''); count_swap_days('Wed', 'Fri')
print('Q8: ',  end=''); count_swap_days('Tue', 'Wed')
print('Q12:',  end=''); count_swap_days('Mon', 'Fri')
print('Q13:',  end=''); count_swap_days('Wed', 'Thu')

---
## 第5节：15道题完整答案一览 / Section 5: Full Answer Table for All 15 Questions

In [ ]:
# Summary of all answers
answers = [
    ('Q1',  'META commission',  '1x100x320x0.001',         32.00,   'commission'),
    ('Q1',  'META short div',   '-(1.50x100x1)x1.01',      -151.50, 'dividend'),
    ('Q2',  'EURUSD swap',      '3x(-4.23)x1x3',           -38.07,  'swap'),
    ('Q3',  'GOOG commission',  '2x100x172x0.001',          34.40,  'commission'),
    ('Q3',  'GOOG long div',    '(2.10x100x2)x0.99',        415.80, 'dividend'),
    ('Q4',  'HK50 commission',  '5x1x18300x0.0035',         320.25, 'commission (HKD)'),
    ('Q5',  'XAGUSD swap',      '2x(-6.00)x3',             -36.00,  'swap'),
    ('Q6',  'BTCUST swap',      '3x98752x1x0.01/360x4',    -32.92,  'swap'),
    ('Q7',  'AMZN commission',  '5x100x129.5x0.001',        64.75,  'commission'),
    ('Q7',  'AMZN short div',   '-(3x100x5)x1.01',         -1515.00,'dividend'),
    ('Q8',  'US30 swap',        '2x39420x1x0.01/360x1',    -2.19,   'swap'),
    ('Q9',  'GBPUSD swap',      '1x(-6.21)x1x7',           -43.47,  'swap'),
    ('Q10', 'TSLA commission',  '3x100x785x0.001',          235.50, 'commission'),
    ('Q10', 'TSLA short div',   '-(1.75x100x3)x1.01',      -530.25, 'dividend'),
    ('Q11', 'Xiaomi commission','10x100x14.6x0.0035',       51.10,  'commission (HKD)'),
    ('Q11', 'Xiaomi short div', '-(0.40x100x10)x1.01/7.82',-51.66,  'dividend (USD)'),
    ('Q12', 'ETHUST swap',      '3x2300x1x0.01/360x6',     -1.15,   'swap'),
    ('Q13', 'NAS100 swap',      '5x(-3.40)x0.01x3',        -0.51,   'swap'),
    ('Q14', 'NFLX commission',  '1x100x400x0.001',          40.00,  'commission'),
    ('Q14', 'NFLX short div',   '-(4x100x1)x1.01',         -404.00, 'dividend'),
    ('Q15', 'JPN225 commission','index CFD has no fee',     0,      'commission'),
    ('Q15', 'JPN225 swap',      '2x(-5.35)x0.01x3',        -0.32,   'swap'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Commission
comm_names = [a[1] for a in answers if 'commission' in a[4] and a[3] != 0]
comm_vals = [a[3] for a in answers if 'commission' in a[4] and a[3] != 0]
axes[0].barh(comm_names, comm_vals, color='#3498db')
axes[0].set_xlabel('USD (or HKD)', fontsize=10)
axes[0].set_title('Commission', fontsize=13, fontweight='bold')
for i, v in enumerate(comm_vals):
    axes[0].text(v + 1, i, f'{v:.1f}', va='center', fontsize=9)

# Dividend
div_names = [a[1] for a in answers if 'dividend' in a[4]]
div_vals = [a[3] for a in answers if 'dividend' in a[4]]
div_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in div_vals]
axes[1].barh(div_names, div_vals, color=div_colors)
axes[1].set_xlabel('USD', fontsize=10)
axes[1].set_title('Dividend (green = received, red = paid)', fontsize=13, fontweight='bold')
axes[1].axvline(x=0, color='black', lw=0.5)

# Swap
swap_names = [a[1] for a in answers if 'swap' in a[4]]
swap_vals = [a[3] for a in answers if 'swap' in a[4]]
axes[2].barh(swap_names, swap_vals, color='#f39c12')
axes[2].set_xlabel('USD', fontsize=10)
axes[2].set_title('Overnight swap', fontsize=13, fontweight='bold')
for i, v in enumerate(swap_vals):
    axes[2].text(v - 2 if abs(v) > 5 else v - 0.5, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 第6节：总结速查表 / Section 6: Cheat Sheet

### 手续费 / Commission
| 市场 / Market | 公式 / Formula | 费率 / Rate |
|------|------|------|
| 美股 / US stocks | 手数 × 100 × 价格 × 0.001 / Lots × 100 × Price × 0.001 | 0.10% |
| 港股 / HK stocks | 手数 × 100 × 价格 × 0.0035 / Lots × 100 × Price × 0.0035 | 0.35% |
| 指数/外汇 / Indices & FX | 无手续费（成本在点差中）/ No commission — cost is inside the spread | - |

### 股息 / Dividend
| 方向 / Direction | 公式 / Formula | 记法 / Mnemonic |
|------|------|------|
| 多单 / Long | +每股股息 × 合约量 × 手数 × **99%** / +DivPerShare × Contract × Lots × **99%** | 收钱，扣1%税 / Receive, 1% tax |
| 空单 / Short | -每股股息 × 合约量 × 手数 × **101%** / -DivPerShare × Contract × Lots × **101%** | 付钱，加1%税 / Pay, 1% tax |

### 隔夜利息 / Overnight Swap
| 模式 / Mode | 公式 / Formula | 适用 / Applies |
|------|------|------|
| 点差模式 / Pip mode | 手数 × 库存费 × 点值 × 天数 / Lots × SwapRate × PipValue × Days | 外汇 / FX |
| 期货模式 / Futures mode | 手数 × 价格 × 合约 × (-1%)/360 × 天数 / Lots × Price × Contract × (-1%)/360 × Days | BTC/ETH/指数 / BTC / ETH / indices |

### 持仓天数 / Days Held
| 核心规则 / Core rule | 周三晚 = **3天**，其余每晚 = **1天** / Wed night = **3 days**, all other nights = **1 day** |
|---------|------|
| 一周五晚 / Full week (5 nights) | 1+1+**3**+1+1 = **7天 / 7 days** |

---

> **回到主章节 / Back to the main chapter**: [`Ch02_margin_liquidation.ipynb`](Ch02_margin_liquidation.ipynb) — 互动保证金模拟器 / interactive margin simulator
>
> **下一章 / Next chapter**: [第三章：交易成本 / Chapter 3: Trading Costs](../Ch03_trading_costs/) — 点差、手续费、隔夜利息 / spread, commission, swap


---

> **回到主章节** → [Ch02 保证金与爆仓](../Ch02_margin_liquidation/Ch02_margin_liquidation.ipynb)
> **下一章** → [第三章：交易成本](../Ch03_trading_costs/) —— 把这三笔费用放进总成本模型里
>
> **Back to the main chapter** → [Ch02 Margin & Liquidation](../Ch02_margin_liquidation/Ch02_margin_liquidation.ipynb)
> **Next chapter** → [Chapter 3: Trading Costs](../Ch03_trading_costs/) — plug these three fees into a total-cost model